# Time dependence

`Model.solve_transient` integrates the population equations from initial densities with a stiff integrator (BDF by default) and the model's sparse Jacobian. The prescribed states of the conditions stay prescribed; the rest evolve. The physics is in [The CR model](/OpenCR/guides/model/), the options in [Solvers and troubleshooting](/OpenCR/guides/solvers/).

In [ ]:
import numpy as np
import plasma_plots

import opencr

model = opencr.load_model("hydrogen/atomic/basic", n_max=12)

## Ionization of a neutral gas

A closed system (no prescribed states, the total atom density given) that starts as cold neutral gas in a plasma of given $T_e$ and $n_e$:

In [ ]:
conditions = opencr.Conditions(3.0, 1e18, totals={"H": 1e19})
times = np.geomspace(1e-9, 1e-1, 90)
history = model.solve_transient(conditions, {"H(n=1)": 1e19}, times)
print(history.report.summary())

The excited states fill within nanoseconds (the quasi-static assumption); the ionization takes tens of milliseconds at this density and temperature:

In [ ]:
data = history.to_dataset()
with plasma_plots.figure(1, 1) as fig:
    for state in ("H(n=1)", "H(n=2)", "H(n=3)", "H+"):
        data["population"].sel(state=state).plasma.plot.lineout(
            ax=fig[0], title="populations against time"
        )
        fig[0].lines[-1].set_label(state)
    fig[0].set_xscale("log")
    fig[0].set_yscale("log")
    fig[0].set_ylim(1e8, 2e19)
    fig[0].set_ylabel("n (m$^{-3}$)")
    fig[0].legend()

The end of the transient is the closed steady state of the same conditions, and any output time can be read like a steady-state result (`history.at(index)`), with the residual saying how far from steady state it is:

In [ ]:
steady = model.solve_steady_state(conditions, mode="full")
final = history.at(-1)
for state in ("H(n=1)", "H(n=2)", "H+"):
    print(
        f"{state:7s} transient {final.population(state):.4e}  steady {steady.population(state):.4e}"
    )
print(f"relative residual at the last time: {final.report.residual:.1e}")
print(final.dominant_processes("H(n=2)", 3))

## Time-dependent conditions

With `conditions` a function of time, the rate coefficients follow it. Here the drivers H(n=1) and H⁺ are prescribed, the electron temperature rises from 1 to 10 eV over 1 µs, and the H-α emission follows the n=3 population:

In [ ]:
base = opencr.Conditions(1.0, 1e18, densities={"H(n=1)": 1e19, "H+": 1e18})


def ramp(t: float) -> opencr.Conditions:
    return base.replace(electron_temperature=1.0 + 9.0 * min(t / 1e-6, 1.0))


times = np.geomspace(1e-9, 1e-5, 60)
heating = model.solve_transient(ramp, {}, times)
h_alpha = np.array(
    [
        heating.at(i).emission()["emissivity"].sel(line="A:3-2").item()
        for i in range(times.size)
    ]
)
with plasma_plots.figure(1, 1) as fig:
    fig[0].loglog(times, h_alpha)
    fig[0].set_xlabel("t (s)")
    fig[0].set_ylabel("H-α emissivity (photons m$^{-3}$ s$^{-1}$)")
    fig[0].set_title("H-α during a temperature ramp")